# 02 — Realisation & Bootstrap GW Spectrum

Load the 2D scan from notebook 01, generate a bubble population,
compute geometric collision weights, and bootstrap the predicted 3D GW spectrum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ptbuilder as pt
from pathlib import Path

## 1. Reload model and scan

In [ ]:
lambda_bar = 0.84
potential  = pt.Phi4Potential(lambda_bar)
config     = pt.Config()
model      = pt.PhysicsModel(potential, config)

# Load cached scan results
cache_path   = model.results_dir / 'scan_cache.h5'
scan_results = pt.load_scan(cache_path)

gammas = np.array(sorted(scan_results.keys()))
times  = next(iter(scan_results.values())).times
print(f"Loaded {len(scan_results)} gamma values, {len(times)} time steps")

## 2. Generate a bubble population

In [ ]:
box_size  = 200.     # box side length (physical units)
n_bubbles = 64
seed      = 42

nucleation = pt.ExponentialNucleation(beta=1.0, t_start=0., t_end=10.)
kinematics = model.kinematics

bubble_pop = pt.BubblePopulation.generate(
    n_bubbles  = n_bubbles,
    nucleation = nucleation,
    kinematics = kinematics,
    box_size   = box_size,
    seed       = seed,
)

print(f"Bubbles:    {bubble_pop.n_bubbles()}")
print(f"Collisions: {bubble_pop.n_collisions()}")
print(f"Gamma_col:  min={bubble_pop.collision_gammas().min():.2f}"
      f"  max={bubble_pop.collision_gammas().max():.2f}")

## 3. Write 3D initial conditions (for validation runs)

Optional: write the bubble injection HDF5 for sledgehamr.

In [ ]:
t_inject  = 15.     # injection time
lattice_N = 64      # base grid
ic_path   = model.results_dir / f'bubbles_N{lattice_N}_t{t_inject:.0f}.h5'

pt.write_3d_bubbles(
    model      = model,
    bubble_pop = bubble_pop,
    t_inject   = t_inject,
    lattice_N  = lattice_N,
    path       = ic_path,
)
print(f"Written sledgehamr IC: {ic_path}")

## 4. Compute geometric collision weights

The weights binary computes, for each colliding pair and each time step,
what fraction of the collision circle perimeter is "unblocked" by other bubbles.

In [ ]:
import h5py

# Write weights input HDF5
weights_in  = model.results_dir / f'weights_in_seed{seed}.h5'
weights_out = model.results_dir / f'weights_out_seed{seed}.h5'

with h5py.File(weights_in, 'w') as f:
    f.attrs['L']   = box_size
    f.attrs['n_t'] = len(times)
    f.attrs['n_b'] = bubble_pop.n_bubbles()
    f.create_dataset('t', data=times)
    # R[i_t] = bubble wall radius at time times[i_t] (both bubbles treated equally)
    R = np.array([kinematics.R(t, 0., 'out') for t in times])
    f.create_dataset('R',     data=R)
    locs = np.array([e.position for e in bubble_pop.events])
    f.create_dataset('xlocs', data=locs[:, 0])
    f.create_dataset('ylocs', data=locs[:, 1])
    f.create_dataset('zlocs', data=locs[:, 2])

# Run weights binary
config.check_binaries()
pt.run_weights(weights_in, weights_out, config)
print(f"Weights written to {weights_out}")

# Load weights and pair indices
weights, pair_i, pair_j = pt.load_weights(weights_out, n_t=len(times))
print(f"Weights shape: {weights.shape}  (n_collisions, n_t)")
print(f"Collision pairs found by weights binary: {len(pair_i)}")

## 5. Bootstrap GW spectrum

In [ ]:
k_out = np.logspace(-2, 1, 200)

k, spec = pt.bootstrap_spectrum(
    scan_results = scan_results,
    bubble_pop   = bubble_pop,
    weights      = weights,
    pair_i       = pair_i,
    pair_j       = pair_j,
    t_range      = (times[0], times[-1]),
    k_out        = k_out,
    lambda_bar   = lambda_bar,
)

plt.figure(figsize=(8, 5))
plt.plot(k, spec)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('k')
plt.ylabel(r'$\mathrm{d}E_{\mathrm{GW}}/\mathrm{d}\ln k$')
plt.title('Bootstrapped GW spectrum')
plt.tight_layout()